# 🧠 การเพิ่มประสิทธิภาพแบบ Momentum และการลดการแกว่งในหุบเหว (Ravine Dampening)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Momentum Optimization**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายหลักฟิสิกส์ของ Momentum (ลูกบอลกลิ้ง) และ Nesterov Accelerated Gradient (การมองไปข้างหน้า/look-ahead)
2. กำหนดฟังก์ชันต้นทุน (Cost Function) แบบหุบเหว 2 มิติที่ลาดชัน:
   $$f(x, y) = 0.5x^2 + 10y^2$$
   โดยที่แกน $y$ มีความชันมากกว่าแกน $x$ ถึง 20 เท่า
3. อิมพลีเมนต์ **Vanilla Gradient Descent**, **Momentum GD** และ **Nesterov Accelerated Gradient (NAG)** จากศูนย์
4. แสดงภาพเส้นทางการลู่เข้าบนแผนที่เส้นชั้นความสูงแบบ 2 มิติ เพื่อสังเกตวิธีที่ Momentum ช่วยลดแรงแกว่งในแนวตั้งฉากและเร่งความเร็วไปตามก้นหุบเขา
5. เปรียบเทียบอัตราการลู่เข้าโดยใช้กราฟแสดงความสัมพันธ์ระหว่างลอสกับจำนวนขั้นตอน
6. เชื่อมโยงกลไกเหล่านี้กับพารามิเตอร์การฝึกตามค่าเริ่มต้นของ YOLO คือ `momentum=0.937`

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลย

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การกำหนดฟังก์ชันหุบเหว (Ravine Function) และเกรเดียนต์

ฟังก์ชันต้นทุนของเราแสดงถึงหุบเขาลึก:
$$f(x, y) = 0.5x^2 + 10y^2$$

เกรเดียนต์:
$$\frac{\partial f}{\partial x} = x, \quad \frac{\partial f}{\partial y} = 20y$$

In [ ]:
def cost_ravine(x, y):
    return 0.5 * x**2 + 10.0 * y**2

def grad_ravine(x, y):
    return np.array([x, 20.0 * y])

## 2. การอิมพลีเมนต์ตัวปรับค่า (Optimizers) จากศูนย์

มาอิมพลีเมนต์ลูปการเพิ่มประสิทธิภาพทั้งสามแบบ:
1.  **Vanilla GD:** $\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \nabla J(\mathbf{w}_t)$
2.  **Momentum:**
    -   $\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \alpha \nabla J(\mathbf{w}_t)$
    -   $\mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{v}_t$
3.  **Nesterov Accelerated Gradient (NAG):**
    -   $\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \alpha \nabla J(\mathbf{w}_t - \beta \mathbf{v}_{t-1})$
    -   $\mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{v}_t$

In [ ]:
def optimize_vanilla(start_pos, lr=0.08, epochs=40):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_momentum(start_pos, lr=0.08, beta=0.9, epochs=40):
    pos = np.array(start_pos, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        v = beta * v + lr * grad
        pos -= v
        history.append(pos.copy())
    return np.array(history)

def optimize_nesterov(start_pos, lr=0.08, beta=0.9, epochs=40):
    pos = np.array(start_pos, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        projected_pos = pos - beta * v
        grad = grad_ravine(projected_pos[0], projected_pos[1])
        v = beta * v + lr * grad
        pos -= v
        history.append(pos.copy())
    return np.array(history)

# Run optimizations starting at (8.0, 4.0)
start = [8.0, 4.0]
path_vanilla = optimize_vanilla(start)
path_momentum = optimize_momentum(start)
path_nesterov = optimize_nesterov(start)

## 3. การแสดงภาพเส้นทางการลู่เข้าบนแผนที่เส้นชั้นความสูง (Contour Map)

มาสร้างกริดเส้นชั้นความสูงแบบ 2 มิติและพล็อตเส้นทางการลู่เข้ากัน

In [ ]:
x = np.linspace(-10, 10, 150)
y = np.linspace(-5, 5, 150)
X, Y = np.meshgrid(x, y)
Z = cost_ravine(X, Y)

plt.figure(figsize=(12, 8))
contours = plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

# Plot paths
plt.plot(path_vanilla[:, 0], path_vanilla[:, 1], color='red', marker='o', alpha=0.8, linewidth=1.5, label='Vanilla GD (Wild wiggles)')
plt.plot(path_momentum[:, 0], path_momentum[:, 1], color='blue', marker='s', linewidth=2.5, label='Momentum GD (Smooth path)')
plt.plot(path_nesterov[:, 0], path_nesterov[:, 1], color='green', marker='x', linewidth=2, linestyle='--', label='Nesterov GD (Projected lookahead)')

plt.scatter(0, 0, color='gold', s=150, marker='*', zorder=5, label='Minimum (0,0)')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(-10, 10)
plt.ylim(-5, 5)
plt.title('Dampening Ravine Oscillations: Vanilla vs. Momentum vs. Nesterov')
plt.legend()
plt.show()

ดูผลลัพธ์กัน!
-   **Vanilla GD (สีแดง):** เนื่องจากความลาดชันของแกน $y$ ชันมาก การอัปเดตจึงเลยเป้าหมาย (overshoot) และกระดอนขึ้นลงตามผนังของหุบเหวเกิดการแกว่งขนาดใหญ่ ทำให้ลู่เข้าตามแนวแกน $x$ ได้ช้ามาก
-   **Momentum (สีน้ำเงิน):** ช่วยลดการแกว่ง! การแกว่งในแนวตั้งฉากจะเฉลี่ยหักล้างกันไป ในขณะที่ความเร็วสะสมตามแนวแกน $x$ จะเสถียรและเร็วขึ้น
-   **Nesterov (สีเขียว):** ลู่เข้าได้เร็วขึ้นไปอีกและลดการเลยเป้าหมาย (overshooting) เนื่องจากมีการเบรกจากฟังก์ชันคาดการณ์ล่วงหน้า (look-ahead correction braking)!

## 4. การเปรียบเทียบความเร็วในการลู่เข้า

ลองพล็อตเส้นกราฟการลดลงของต้นทุนกัน

In [ ]:
cost_vanilla = [cost_ravine(p[0], p[1]) for p in path_vanilla]
cost_momentum = [cost_ravine(p[0], p[1]) for p in path_momentum]
cost_nesterov = [cost_ravine(p[0], p[1]) for p in path_nesterov]

plt.figure(figsize=(10, 5))
plt.plot(cost_vanilla, color='red', label='Vanilla GD')
plt.plot(cost_momentum, color='blue', label='Momentum GD')
plt.plot(cost_nesterov, color='green', label='Nesterov GD')
plt.yscale('log')
plt.xlabel('Steps')
plt.ylabel('Log Cost')
plt.title('Cost Convergence Comparison (Log Scale)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 💡 ความเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **YOLO SGD Momentum:** เมื่อทำการฝึกโมเดล YOLO ค่าพารามิเตอร์ momentum เริ่มต้นจะถูกตั้งค่าเป็น `0.937` (ซึ่งกำหนดไว้ในไฟล์คอนฟิกสำหรับการฝึก) หากค่าแรงส่งนี้ต่ำเกินไป (เช่น `0.0`) โมเดลจะใช้เวลาฝึกนานกว่ามาก หากค่าแรงส่งสูงเกินไป (เช่น `0.999`) ความเร็วในการกลิ้งสะสมของโมเดลจะสูงมากจนเลยจุดน้ำหนักที่เหมาะสมที่สุดและเกิดการลู่ออก (diverge)
*   **พารามิเตอร์การมองล่วงหน้า:** YOLO อนุญาตให้กำหนดค่าพารามิเตอร์อื่นๆ เช่น weight decay เพื่อทำงานร่วมกับ momentum ในการควบคุมขนาดของค่าน้ำหนักให้มีเสถียรภาพ